# Parallel batch extraction and Parquet output

`extract_features_from_folder` with `n_jobs` runs files in worker processes, with output identical to a serial run and in the same order. Parquet output has one row per file and one named column per value (`mfcc_0`, ..., `zcr`) and loads straight into pandas.

The same from the shell: `afx <folder> --output features.parquet -j 8`

In [ ]:
import json
import os
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

from AFX.batch_extractor import extract_features_from_folder

SR = 16000

## Build a working folder

Up to 12 clips from the environmental-sound dataset named by an environment variable (Nonspeech7k, ESC-50 or UrbanSound8K), otherwise synthesized clips.

In [ ]:
import glob

# Environmental-sound dataset roots are passed via environment variables.
# Set any ONE of these before launching Jupyter, e.g.
#   export ESC_50_dataset=/path/to/ESC-50
DATASET_ENV_VARS = ['dataset_nonspeech7k', 'nonspeech7k_dataset',
                    'ESC_50_dataset', 'UrbanSound8K_dataset']

def find_dataset_audio(max_files=None, exts=('.wav', '.ogg', '.flac')):
    """Return audio file paths from the first configured dataset env var
    (empty list if no dataset is configured)."""
    for var in DATASET_ENV_VARS:
        root = os.environ.get(var)
        if root and os.path.isdir(root):
            files = sorted(
                p for ext in exts for p in
                glob.glob(os.path.join(root, '**', '*' + ext), recursive=True)
            )
            if files:
                print(f'Dataset: {var} -> {root} ({len(files)} audio files)')
            return files[:max_files] if max_files else files
    return []

def synth_clip(kind, seconds=1.0, sr=22050, seed=0):
    """Synthesize a simple environmental-sound-like clip (fallback when no
    dataset env var is configured)."""
    rng = np.random.default_rng(seed)
    t = np.arange(int(sr * seconds)) / sr
    if kind == 'tonal':        # bird-chirp-like rising tone
        f = 800 + 600 * t
        sig = 0.6 * np.sin(2 * np.pi * f * t)
    elif kind == 'noisy':      # rain-like broadband noise
        sig = 0.4 * rng.standard_normal(t.size)
    else:                      # engine-like low harmonic stack
        sig = sum(0.3 / k * np.sin(2 * np.pi * 90 * k * t) for k in (1, 2, 3))
    sig = sig + 0.02 * rng.standard_normal(t.size)
    return sig.astype(np.float32)

workdir = Path(tempfile.mkdtemp(prefix='afx_batch_'))
audio_dir = workdir / 'audio'
audio_dir.mkdir()

files = find_dataset_audio(max_files=12)
if len(files) >= 16:
    for f in files:
        shutil.copy(f, audio_dir / os.path.basename(f))
else:
    print('No (large enough) dataset configured - synthesizing 12 clips.')
    kinds = ['tonal', 'noisy', 'harmonic']
    for i in range(12):
        sf.write(audio_dir / f'clip_{i:02d}.wav',
                 synth_clip(kinds[i % 3], seconds=3.0, sr=SR, seed=i), SR)

config = {
    'sample_rate': SR,
    'features': {
        'mfcc': {'n_mfcc': 13, 'frame_size': 1024, 'hop_length': 512},
        'mfcc_delta': {'n_mfcc': 13, 'frame_size': 1024, 'hop_length': 512},
        'melspectrogram': {'n_mels': 64, 'frame_size': 1024, 'hop_length': 512},
        'chroma_stft': {'frame_size': 1024, 'hop_length': 512},
        'spectral_contrast': {'frame_size': 1024, 'hop_length': 512},
        'spectral_centroid': {'frame_size': 1024, 'hop_length': 512},
        'spectral_flatness': {'frame_size': 1024, 'hop_length': 512},
        'sample_entropy': {'frame_size': 2048, 'hop_length': 4096},
        'zcr': {'frame_size': 1024, 'hop_length': 512},
    },
    'aggregation': 'mean',
}
config_path = workdir / 'config.json'
config_path.write_text(json.dumps(config, indent=2))
print(f'{len(list(audio_dir.iterdir()))} files in {audio_dir}')

## Serial vs parallel

In [ ]:
def timed_run(n_jobs, tag):
    out = workdir / f'features_{tag}.json'
    t0 = time.perf_counter()
    res = extract_features_from_folder(
        str(audio_dir), str(config_path), str(out),
        save_format='json', n_jobs=n_jobs,
    )
    dt = time.perf_counter() - t0
    return res, dt

n_workers = os.cpu_count()
res_serial, t_serial = timed_run(1, 'serial')
res_parallel, t_parallel = timed_run(0, 'parallel')   # 0 = all cores

print(f'serial:   {t_serial:.2f} s')
print(f'parallel: {t_parallel:.2f} s  (n_jobs={n_workers}, '
      f'speedup x{t_serial / t_parallel:.2f})')

# Parallel results are identical to serial, in identical order
assert list(res_serial) == list(res_parallel)
for k in res_serial:
    for feat in res_serial[k]:
        np.testing.assert_allclose(res_serial[k][feat], res_parallel[k][feat])
print('parallel output identical to serial: OK')

plt.figure(figsize=(4.5, 3))
plt.bar(['serial', f'n_jobs={n_workers}'], [t_serial, t_parallel],
        color=['#888', '#4C72B0'])
plt.ylabel('seconds')
plt.title('Batch extraction wall time')
plt.tight_layout()

> Worker startup is a fixed cost, so the speedup approaches the core count only on thousands of files. `n_jobs=0` uses all cores.

## Parquet output

One row per file, vector features expanded to named columns. Needs `pip install "afx-audio[parquet]"`; the cell falls back to CSV without pyarrow.

In [ ]:
try:
    import pyarrow  # noqa: F401
    out_path = workdir / 'features.parquet'
    extract_features_from_folder(
        str(audio_dir), str(config_path), str(out_path),
        save_format='parquet', n_jobs=0,
    )
    df = pd.read_parquet(out_path)
except ImportError:
    print('pyarrow not installed - falling back to CSV.')
    out_path = workdir / 'features.csv'
    extract_features_from_folder(
        str(audio_dir), str(config_path), str(out_path),
        save_format='csv', n_jobs=0,
    )
    df = pd.read_csv(out_path, index_col='file')

print(out_path.name, df.shape)
df.head()

In [ ]:
# Feature table ready for ML: named columns, one row per file
df[['mfcc_0', 'mfcc_1', 'spectral_centroid', 'spectral_flatness',
    'zcr']].describe().T[['mean', 'std', 'min', 'max']]

## Notes

- Failed files never poison the batch: errors are collected into `<output>.errors.json` and reported, while successful rows are written normally.
- The same options are exposed on the CLI: `--jobs/-j`, `--format parquet`, `--features`, `--aggregation`, `--normalize`.
- For downstream training, `pd.read_parquet(...)` + `df.to_numpy()` (or polars/duckdb) makes the extracted table immediately usable.

In [ ]:
# Clean up the temporary working folder
shutil.rmtree(workdir)
print('removed', workdir)